# Resolving a Lookahead Conflict

The notebook [03-Look-Ahead.ipynb](03-Look-Ahead.ipynb) has discussed the grammar
```
    a : b "U" "V"
      | c "U" "W"

    b : "X"

    c : "X"
```
which is *unambiguous* but nevertheless not an `LALR(1)` grammar:  after an `'X'` has been read and the next
token is a `'U'`, the parser cannot decide whether to reduce with the rule for `b` or with the rule for `c`,
since that is only settled by the token *following* the `'U'`.  `Lark` therefore refuses to build an `LALR`
parser and raises a `GrammarError`.

The problem is that the grammar tries to interpret an `'X'` either as a `b` or as a `c` depending on the
context.  The remedy is to pull the context `'U'`, which follows both `b` and `c`, into the rules for `b`
and `c`:
```
    a : b "V"
      | c "W"

    b : "X" "U"

    c : "X" "U"
```
This grammar generates the same language, but now the decision between `b` and `c` is postponed until the
decisive token is the *next* token.  We will see that there are no conflicts.

## Specification of the Grammar

In [1]:
grammar = r"""
    a : b "V"
      | c "W"

    b : "X" "U"

    c : "X" "U"
"""

## Specification of the Parser

We lower the level of the logger `lark.logger` and create the parser with `debug=True`, for otherwise `Lark`
would keep silent about any conflicts:  the messages are written to this logger and its level is set to
`logging.CRITICAL` when `Lark` is imported.

Since **no** warning is printed by the cell below, the grammar has no conflicts and is therefore an
`LALR(1)` grammar.

The keyword argument `keep_all_tokens=True` tells `Lark` to keep the anonymous terminals in the parse tree.
Without it, the tokens would be filtered out and the trees at the end of this notebook would be almost empty.

In [2]:
import logging

from lark import Lark, logger

logger.setLevel(logging.DEBUG)

In [3]:
parser = Lark(grammar, start='a', parser='lalr', debug=True, keep_all_tokens=True)

## Inspecting the LALR States

As the parser is created with `debug=True`, the parse table is kept in a form where the states are still the *sets of marked
rules* that we have discussed in the lecture.  The function `dump_states` is the same one that we have used
in [01-Conflicts.ipynb](01-Conflicts.ipynb).

Since the states are sets, iterating over them is not reproducible from one run to the next.  Therefore, the
states are sorted before they are numbered and the start state is put first.

In [4]:
from lark.parsers.lalr_analysis import Shift

def dump_states(lark_parser):
    table  = lark_parser.parser.parser.parser.parse_table
    key    = lambda state: sorted(str(rule) for rule in state)
    start  = next(iter(table.start_states.values()))
    rest   = sorted((s for s in table.states if s != start), key=key)
    order  = [start] + rest
    number = { state: i for i, state in enumerate(order) }
    for state in order:
        print(f'state {number[state]}')
        for marked_rule in sorted(state, key=str):
            print(f'    {marked_rule}')
        print()
        for token, (action, arg) in sorted(table.states[state].items()):
            if action is Shift:
                print(f'    {token:<8} shift and go to state {number[arg]}')
            else:
                print(f'    {token:<8} reduce using rule {arg}')
        print()

In [5]:
dump_states(parser)

state 0
    <$root_a :  * a>
    <a :  * b V>
    <a :  * c W>
    <b :  * X U>
    <c :  * X U>

    X        shift and go to state 6
    a        shift and go to state 1
    b        shift and go to state 2
    c        shift and go to state 4

state 1
    <$root_a : a * >


state 2
    <a : b * V>

    V        shift and go to state 3

state 3
    <a : b V * >

    $END     reduce using rule <a : b V>

state 4
    <a : c * W>

    W        shift and go to state 5

state 5
    <a : c W * >

    $END     reduce using rule <a : c W>

state 6
    <b : X * U>
    <c : X * U>

    U        shift and go to state 7

state 7
    <b : X U * >
    <c : X U * >

    V        reduce using rule <b : X U>
    W        reduce using rule <c : X U>



The interesting state is the one that contains the two marked rules
```
    <b : X U * >
    <c : X U * >
```
This is the state that corresponds to the state which caused the conflict in the previous notebook.  In the
notation of the lecture notes it has the form
$$ \{\; b \rightarrow \texttt{'X'}\,\texttt{'U'} \bullet : \texttt{'V'}, \quad
       c \rightarrow \texttt{'X'}\,\texttt{'U'} \bullet : \texttt{'W'} \;\} $$
and the follow sets of the two rules are now *disjoint*.  Therefore the action table of this state offers
exactly one reduce action per token:  a `'V'` reduces with the rule for `b`, while a `'W'` reduces with the
rule for `c`.  No other state contains more than one marked rule whose position is at the very end, so there
is no conflict anywhere.

Note that `Lark` marks the position of the parser inside a rule with the character `*` instead of the bullet
`•` that we use in the lecture notes, and that the end of the input is called `$END`.

## Parsing Some Strings

Finally, we check that the parser really distinguishes a `b` from a `c`.

In [6]:
print(parser.parse('XUV').pretty())

a
  b
    X
    U
  V



In [7]:
print(parser.parse('XUW').pretty())

a
  c
    X
    U
  W



A string that is not derivable from `a` is rejected.  There is no need for a function like `p_error`:
`Lark` raises an exception of class `UnexpectedInput` whose message already reports the position and the
tokens that would have been acceptable.

In [8]:
from lark.exceptions import UnexpectedInput

try:
    parser.parse('XUZ')
except UnexpectedInput as e:
    print(e)

No terminal matches 'Z' in the current parser context, at line 1 col 3

XUZ
  ^
Expected one of: 
	* V
	* W

Previous tokens: Token('U', 'U')

